<a href="https://colab.research.google.com/github/TrajanDS/SFT_Agent_POC/blob/main/test_gemini_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import google.generativeai as genai
from google.colab import userdata
from tqdm.notebook import tqdm
import pandas as pd
from google.colab import drive

In [ ]:
# Mount Google Drive
### (when prompted, approve all permissions and continue)
drive.mount('/content/drive')

# Fetch your API key from Colab Secrets
#GOOGLE_API_KEY = userdata.get('')
genai.configure(api_key='INPUT API KEY HERE')

# Initialize the Gemini model
gemini_model = genai.GenerativeModel('models/gemini-2.5-flash')

# Set output directory (local) path for images and on drive
drive_output_dir = "/content/drive/MyDrive/CFG_complaints/raw_data"

# Read in CFG complaints dataset
comp_df = pd.read_parquet(f"{drive_output_dir}/cfpb_complaints.parquet")

In [ ]:
def classify_product_with_gemini(
    complaint_row,
    product_types = [
        'Checking or savings account',
        'Money transfer, virtual currency, or money service',
        'Vehicle loan or lease',
        'Credit card',
        'Payday loan, title loan, personal loan, or advance loan',
        'Credit reporting or other personal consumer reports',
        'Mortgage',
        'Student loan',
        'Debt collection',
        'Debt or credit management'
    ]
    ):
    prompt = f"""You are an expert financial complaint analyst. Your task is to classify the product type of the following consumer complaint. Based on the provided details, identify the most appropriate 'product' category from the predefined list.

Predefined Product Categories: {product_types}

Complaint Details:
Date Received: {complaint_row['date_received']}
Issue: {complaint_row['issue']}
Sub-issue: {complaint_row['sub_issue']}
Company: {complaint_row['company']}
Complaint Narrative: {complaint_row['complaint_what_happened']}

Based on these details, what is the product category for this complaint? Please respond with ONLY the product category string. If you cannot determine a specific product, respond with 'Unknown'."""

    try:
        # Using generate_content for single turn interaction
        response = gemini_model.generate_content(prompt)
        # Accessing response.text assumes the model provides plain text output
        # If the model is fine-tuned to output JSON, you might parse response.text as JSON
        return response.text.strip()
    except Exception as e:
        print(f"Error classifying complaint ID {complaint_row['complaint_id']}: {e}")
        return "Error_Classification"

In [ ]:
tqdm.pandas() # Initialize tqdm for pandas

# Inference gemini model to classify the different product types
gem_inf_df = comp_df.head(5).progress_apply(classify_product_with_gemini, axis=1)

In [ ]:
# Display the value counts of the Gemini-classified products
print("Gemini Classified Product Value Counts:")
display(gem_inf_df['gemini_classified_product'].value_counts())

# Display a comparison of original vs. Gemini classified for a few rows
print("\nComparison of Original vs. Gemini Classified Products (first 10 rows):")
display(gem_inf_df[['product', 'gemini_classified_product', 'complaint_what_happened']].head(10))

# Optionally, calculate accuracy (if original labels are considered ground truth)
correct_classifications = (gem_inf_df['product'] == df['gemini_classified_product']).sum()
total_classifications = len(gem_inf_df)
accuracy = correct_classifications / total_classifications
print(f"\nClassification Accuracy (vs. original 'product' column): {accuracy:.2f}")